# Level 10 — Digital Twin: Tangki Air

Notebook ini membuat digital twin lokal tanpa perangkat fisik: plant fisik simulasi, sensor noisy/dropout, state estimator, residual anomaly detection, dan what-if. Plant, sensor, dan twin sengaja dipisah agar setiap bagian dapat diuji.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
DT = 1.0
AREA = 10.0
N_STEPS = 500

## 1. Plant fisik, sensor, dan twin

`step_plant` menghasilkan state fisik tersembunyi. Sensor mengamati state dengan noise dan dropout. Twin hanya melihat input pompa dan observasi sensor—bukan state plant secara langsung.

In [ ]:
def step_plant(level, inflow, coefficient, area=AREA, dt=DT):
    outflow = coefficient * np.sqrt(max(level, 0.0))
    next_level = level + dt / area * (inflow - outflow)
    return max(0.0, next_level), outflow

def observe(level, rng, noise_std=0.05, dropout_probability=0.05, bias=0.0):
    if rng.random() < dropout_probability:
        return np.nan
    return level + bias + rng.normal(0, noise_std)

def step_twin(previous_estimate, inflow, coefficient, measurement, gain=0.35):
    predicted, _ = step_plant(previous_estimate, inflow, coefficient)
    if np.isnan(measurement):
        return predicted, predicted
    estimate = predicted + gain * (measurement - predicted)
    return estimate, predicted

## 2. Simulasi normal lalu fault

Sampai langkah 299, plant dan twin memakai koefisien outflow sama. Mulai langkah 300, plant mengalami clog sehingga outflow nyata lebih kecil. Twin belum mengetahui perubahan itu; residual harus meningkat.

In [ ]:
true_level = 2.0
twin_level = 2.0
rows = []

for t in range(N_STEPS):
    inflow = 1.2 + 0.3 * np.sin(t / 30)
    true_coefficient = 0.55 if t < 300 else 0.40
    true_level, outflow = step_plant(true_level, inflow, true_coefficient)
    measured = observe(true_level, rng)
    twin_level, predicted = step_twin(twin_level, inflow, 0.55, measured)
    rows.append({
        'step': t, 'inflow': inflow, 'true_level': true_level,
        'measured_level': measured, 'predicted_level': predicted,
        'twin_level': twin_level, 'residual': measured - predicted,
    })

df = pd.DataFrame(rows)
print(df.head())
print('Sensor dropout:', df['measured_level'].isna().sum())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df['step'], df['true_level'], label='Plant state (ground truth)', linewidth=2)
axes[0].plot(df['step'], df['measured_level'], '.', label='Sensor', alpha=0.35)
axes[0].plot(df['step'], df['twin_level'], label='Twin estimate', linewidth=2)
axes[0].axvline(300, color='crimson', linestyle='--', label='Clog fault')
axes[0].set_ylabel('Level (m)')
axes[0].set_title('Plant, sensor, dan state twin')
axes[0].legend()
axes[1].plot(df['step'], df['residual'], label='Sensor - prediction')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].axvline(300, color='crimson', linestyle='--')
axes[1].set(xlabel='Step', ylabel='Residual (m)', title='Residual sebelum correction')
plt.tight_layout()

## 3. Alarm residual

Threshold dihitung dari periode normal saja (`step < 250`). Residual setelah correction tidak dipakai karena correction dapat menyembunyikan fault. Alarm adalah sinyal investigasi, bukan diagnosis otomatis.

In [ ]:
normal = df.loc[df['step'] < 250, 'residual'].dropna()
residual_center = normal.mean()
residual_spread = normal.std(ddof=1)
threshold = 3 * residual_spread
df['alarm'] = (df['residual'] - residual_center).abs() > threshold

first_alarm = df.loc[(df['step'] >= 300) & df['alarm'], 'step'].min()
false_alarms = df.loc[df['step'] < 300, 'alarm'].sum()
print(f'Threshold: ±{threshold:.3f} m around {residual_center:.3f}')
print(f'First alarm after fault: {first_alarm}')
print(f'False alarms before fault: {false_alarms}')

plt.figure(figsize=(12, 4))
plt.plot(df['step'], df['residual'], label='Residual')
plt.axhline(residual_center + threshold, color='crimson', linestyle='--', label='Alarm threshold')
plt.axhline(residual_center - threshold, color='crimson', linestyle='--')
plt.scatter(df.loc[df['alarm'], 'step'], df.loc[df['alarm'], 'residual'],
            color='crimson', s=18, label='Alarm')
plt.legend(); plt.xlabel('Step'); plt.ylabel('Residual (m)'); plt.tight_layout()

## 4. What-if tanpa mengubah aset

Fungsi di bawah menggunakan state twin saat ini dan memproyeksikan level untuk skenario inflow berbeda. Ia hanya mengembalikan prediksi; tidak ada command ke plant fisik.

In [ ]:
def simulate_scenario(initial_level, inflow, coefficient=0.55, steps=60):
    levels = [initial_level]
    level = initial_level
    for _ in range(steps):
        level, _ = step_plant(level, inflow, coefficient)
        levels.append(level)
    return np.array(levels)

current_estimate = float(df['twin_level'].iloc[-1])
scenarios = {'pump_off': 0.0, 'normal': 1.2, 'high': 2.0}
for name, inflow in scenarios.items():
    projection = simulate_scenario(current_estimate, inflow)
    plt.plot(projection, label=name)
plt.axhline(1.0, color='crimson', linestyle='--', label='minimum safe level')
plt.axhline(5.0, color='crimson', linestyle='--', label='maximum safe level')
plt.legend(); plt.xlabel('Future step'); plt.ylabel('Projected level (m)'); plt.tight_layout()

## Eksperimen lanjutan

1. Ubah `gain` menjadi 0.1 dan 0.8; bandingkan MAE dan respons noise.
2. Ganti clog dengan sensor bias positif; apakah residual terlihat sama?
3. Tambahkan quality flag untuk dropout dan jangan mengirim command ketika sensor stale.
4. Buat controller hanya sebagai rekomendasi, dengan hard safety bounds dan manual approval.